In [ ]:
feature_cols = [c for c in battery_health.columns if c not in ['battery_id','cycle','SoH']]

# build arrays
X_list, y_list, groups = [], [], []
for (bid, cyc), grp in battery_health.groupby(['battery_id','cycle']):
    X_list.append(grp[feature_cols].values.T.astype(np.float32))
    y_list.append(grp['SoH'].iloc[0] / 100.0)
    groups.append(bid)
X = np.stack(X_list)
y = np.array(y_list, dtype=np.float32)
groups = np.array(groups)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, va_idx = next(gss.split(X, y, groups=groups))
X_tr, X_va = X[tr_idx], X[va_idx]
y_tr, y_va = y[tr_idx], y[va_idx]

mu, sd = X_tr.mean((0,2), keepdims=True), X_tr.std((0,2), keepdims=True)
X_tr = (X_tr - mu) / (sd + 1e-6)
X_va = (X_va - mu) / (sd + 1e-6)

train_ds = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
val_ds = TensorDataset(torch.tensor(X_va), torch.tensor(y_va))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

class CNNRegressor(nn.Module):
    def __init__(self, in_ch, depth=2, width=64, kernel=3, p=0.1):
        super().__init__()
        layers, ch = [], in_ch
        for i in range(depth):
            layers += [
                nn.Conv1d(ch, width, kernel_size=kernel, padding=kernel//2),
                nn.BatchNorm1d(width),
                nn.ReLU()
            ]
            if i < depth - 1:
                layers.append(nn.Dropout(p))
            ch = width
        layers.append(nn.AdaptiveAvgPool1d(1))
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(width, 1)

    def forward(self, x):
        x = self.features(x).squeeze(-1)
        return self.head(x).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNRegressor(len(feature_cols)).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
criterion = nn.L1Loss() 

#early stopping
best_val = 1e9
no_imp = 0
train_maes, val_maes = [], []

for epoch in range(50):
    # train
    model.train()
    acc = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        acc += loss.item() * xb.size(0)
    train_mae = acc / len(train_loader.dataset)
    train_maes.append(train_mae)

    model.eval()
    acc = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            acc += criterion(model(xb), yb).item() * xb.size(0)
    val_mae = acc / len(val_loader.dataset)
    val_maes.append(val_mae)
    scheduler.step(val_mae)

    if val_mae < best_val - 1e-4:
        best_val = val_mae
        no_imp = 0
        torch.save(model.state_dict(), "best_soh_model.pt")
    else:
        no_imp += 1
        if no_imp >= 20:
            break

    if epoch == 1 or epoch % 10 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:02d} | Train MAE: {train_mae*100:.2f}% | Val MAE: {val_mae*100:.2f}% | LR: {lr:.1e}")

plt.plot(np.array(train_maes)*100, label="Train")
plt.plot(np.array(val_maes)*100, label="Val")
plt.xlabel("Epoch"); plt.ylabel("MAE (%)"); plt.legend(); plt.show()

model.load_state_dict(torch.load("best_soh_model.pt"))
model.eval()
with torch.no_grad():
    y_pred = model(torch.tensor(X_va).to(device)).cpu().numpy() * 100
y_true = y_va * 100

final_mae = mean_absolute_error(y_true, y_pred)
print(f"Final Val MAE: {final_mae:.2f}%")

plt.figure(figsize=(5,5))
plt.scatter(y_true, y_pred, alpha=0.6)
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
plt.plot(lims, lims, 'k--')
plt.xlabel("True SoH (%)")
plt.ylabel("Predicted SoH (%)")
plt.title("True vs Predicted SoH")
plt.show()

In [ ]:
#CNN using inverse-sqrt bin weighting to try and improve lower SoH predictions

#normalized cycle index
if "cycle_norm" not in battery_health.columns:
    eol = full_df.groupby("battery_id")["cycle"].max().rename("eol_cycle")
    battery_health = battery_health.merge(eol, on="battery_id")
    battery_health["cycle_norm"] = battery_health["cycle"] / battery_health["eol_cycle"]

features = ["Voltage_measured","Current_measured","Temperature_measured", "SoC","ambient_temperature","cycle_norm"]
X_list, y_list, groups = [], [], []
for (bid, cyc), grp in battery_health.groupby(["battery_id","cycle"]):
    X_list.append(grp[features].values.T.astype(np.float32))
    y_list.append(grp["SoH"].iloc[0] / 100.0)
    groups.append(bid)
X = np.stack(X_list)
y = np.array(y_list, dtype=np.float32)
groups = np.array(groups)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, va_idx = next(gss.split(X, y, groups=groups))
X_tr, X_va = X[tr_idx], X[va_idx]
y_tr, y_va = y[tr_idx], y[va_idx]

mu, sd = X_tr.mean((0,2), keepdims=True), X_tr.std((0,2), keepdims=True)
X_tr = (X_tr - mu) / (sd + 1e-6)
X_va = (X_va - mu) / (sd + 1e-6)

train_ds = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
val_ds = TensorDataset(torch.tensor(X_va), torch.tensor(y_va))

#compute inverse-sqrt bin weights for SoH distribution
n_bins = 5
bin_ids = np.digitize(y_tr, np.linspace(0,1,n_bins+1)) - 1
counts = np.bincount(bin_ids, minlength=n_bins)
bin_weights = 1.0 / np.sqrt(counts + 1e-6)
sample_weights = bin_weights[bin_ids]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

#same parameters as before
class CNNRegressor(nn.Module):
    def __init__(self, in_ch=6, depth=2, width=64, kernel=3, dropout=0.1):
        super().__init__()
        layers, ch = [], in_ch
        for i in range(depth):
            layers += [
                nn.Conv1d(ch, width, kernel, padding=kernel//2),
                nn.BatchNorm1d(width),
                nn.ReLU()
            ]
            if i < depth-1:
                layers.append(nn.Dropout(dropout))
            ch = width
        layers.append(nn.AdaptiveAvgPool1d(1))
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(width, 1)

    def forward(self, x):
        x = self.features(x).squeeze(-1)
        return self.head(x).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNRegressor().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, factor=0.5)
criterion = nn.L1Loss()
train_mae, val_mae = [], []

for epoch in range(60):
    model.train()
    total, count = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = (torch.abs(model(xb) - yb) * torch.tensor(bin_weights[bin_ids][count:count+xb.size(0)], device=device)).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * xb.size(0)
        count += xb.size(0)
    train_mae.append(total / count)

    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            total += torch.abs(model(xb) - yb).sum().item()
            count += xb.size(0)
    val_mae.append(total / count)
    scheduler.step(val_mae[-1])

    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:02d} | Train MAE: {train_mae[-1]*100:.2f}% | Val MAE: {val_mae[-1]*100:.2f}%")

plt.plot(np.array(train_mae)*100, label="Train")
plt.plot(np.array(val_mae)*100, label="Val")
plt.xlabel("Epoch")
plt.ylabel("MAE (%)")
plt.legend()
plt.show()
model.eval()
with torch.no_grad():
    y_pred = model(torch.tensor(X_va).to(device)).cpu().numpy() * 100
y_true = y_va * 100
print("Final Val MAE: {:.2f}%".format(mean_absolute_error(y_true, y_pred)))
plt.figure(figsize=(5,5))
plt.scatter(y_true, y_pred, alpha=0.6)
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
plt.plot(lims, lims, 'k--')
plt.xlabel("True SoH (%)")
plt.ylabel("Predicted SoH (%)")
plt.title("True vs Predicted SoH")
plt.show()

In [ ]:
#Preparing RUL df

eol = full_df.groupby("battery_id")["cycle"].max().rename("eol_cycle") #end of life per battery

# collapse each battery and cycle to one row of summary stats + RUL
records = []
for (bid, cyc), grp in full_df.groupby(["battery_id","cycle"]):
    records.append({"battery_id": bid,"cycle": cyc,"rul": eol.loc[bid] - cyc,"volt_mean": grp.Voltage_measured.mean(),
        "volt_std": grp.Voltage_measured.std(),"curr_mean": grp.Current_measured.mean(),"curr_std": grp.Current_measured.std(),
        "temp_mean": grp.Temperature_measured.mean(),"temp_std": grp.Temperature_measured.std()})
df_rul = pd.DataFrame(records)

df_soh = (battery_health.drop_duplicates(["battery_id","cycle"]).loc[:,["battery_id","cycle","SoH"]].rename(columns={"SoH":"SoH_pct"}))
df_rul = df_rul.merge(df_soh, on=["battery_id","cycle"], how="left")
df_rul = df_rul.dropna(subset=["SoH_pct"]).reset_index(drop=True)
df_rul["eol_cycle"]  = df_rul.groupby("battery_id")["cycle"].transform("max")
df_rul["cycle_frac"] = df_rul["cycle"] / df_rul["eol_cycle"]

feature_cols_ext = ["volt_mean","volt_std","curr_mean","curr_std","temp_mean","temp_std","cycle_frac","SoH_pct"]
X = df_rul[feature_cols_ext].values.astype(np.float32)
y = df_rul["rul"].values.astype(np.float32)


In [ ]:
#Group Split, Standard Scaler and DataLoaders

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df_rul["battery_id"]))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
X_val, X_te, y_val, y_te = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_te = scaler.transform(X_te)

def make_loader(X, y, bs=32, shuffle=True):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, bs=32, shuffle=True)
val_loader = make_loader(X_val, y_val, bs=64, shuffle=False)
test_loader = make_loader(X_te, y_te, bs=64, shuffle=False)

In [ ]:
class RULExtRegressor(nn.Module):
    def __init__(self, in_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_ext = RULExtRegressor(len(feature_cols_ext), 128).to(device)
opt_ext = optim.AdamW(model_ext.parameters(), lr=5e-4, weight_decay=1e-4)
criterion = nn.L1Loss()

train_mae, val_mae = [], []
for ep in range(60):
    model_ext.train()
    tot = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt_ext.zero_grad()
        loss = criterion(model_ext(xb), yb)
        loss.backward()
        opt_ext.step()
        tot += loss.item() * xb.size(0)
    train_mae.append(tot / len(train_loader.dataset))

    model_ext.eval()
    tot = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            tot += criterion(model_ext(xb), yb).item() * xb.size(0)
    val_mae.append(tot / len(val_loader.dataset))

    if ep % 10 == 0:
        print(f"Epoch {ep:02d} | Train MAE: {train_mae[-1]:.4f} | Val MAE: {val_mae[-1]:.4f}")

plt.plot(train_mae, label="Train MAE")
plt.plot(val_mae, label="Val MAE")
plt.xlabel("Epoch"); plt.ylabel("MAE"); plt.legend(); plt.show()

model_ext.eval()
with torch.no_grad():
    y_pred = model_ext(torch.tensor(X_te, dtype=torch.float32, device=device)).cpu().numpy()

test_mae = mean_absolute_error(y_te, y_pred)
test_mae_pct = test_mae / y_te.mean() * 100
print(f"Extended RUL Test MAE: {test_mae:.2f} cycles ({test_mae_pct:.2f}%)")

plt.figure(figsize=(6,6))
plt.scatter(y_te, y_pred, alpha=0.6)
lims = [min(y_te.min(), y_pred.min()), max(y_te.max(), y_pred.max())]
plt.plot(lims, lims, 'k--')
plt.xlabel("True RUL"); plt.ylabel("Predicted RUL")
plt.title("True vs Predicted RUL (Test)"); plt.show()

In [ ]:
#Grid Search for RUL MLP Hyperparameters

hiddens = [64, 128, 256]
dropouts = [0.1, 0.3, 0.5]
lrs = [1e-3, 5e-4]
wds = [1e-5, 1e-4]
param_grid = list(itertools.product(hiddens, dropouts, lrs, wds))

class MLP_RUL(nn.Module):
    def __init__(self, in_dim, hidden, drop):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(hidden, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
results = []

for hidden, drop, lr, wd in param_grid:
    model = MLP_RUL(in_dim=len(feature_cols_ext), hidden=hidden, drop=drop).to(device)
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    crit = nn.MSELoss()

    for epoch in range(30):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()

    model.eval()
    val_preds = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            val_preds.append(model(xb).cpu().numpy())
    val_preds = np.concatenate(val_preds)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))

    results.append({"hidden": hidden,"dropout": drop, "lr": lr,"wd": wd, "val_rmse": val_rmse})


df_grid = pd.DataFrame(results)
display(df_grid.sort_values("val_rmse").reset_index(drop=True))

In [ ]:
#Heatmap of Grid Search

lrs = [1e-3, 5e-4]
wds = [1e-5, 1e-4]

for lr in lrs:
    for wd in wds:
        sub = df_grid[(df_grid.lr == lr) & (df_grid.wd == wd)]
        if sub.empty: 
            continue
        pivot = sub.pivot(index='hidden', columns='dropout', values='val_rmse')
        plt.figure(figsize=(5,4))
        plt.title(f"Val RMSE (lr={lr}, wd={wd})")
        plt.imshow(pivot, origin='lower', aspect='auto')
        plt.xticks(np.arange(len(pivot.columns)), pivot.columns)
        plt.yticks(np.arange(len(pivot.index)), pivot.index)
        plt.xlabel("Dropout")
        plt.ylabel("Hidden Size")
        plt.colorbar(label="Val RMSE (cycles)")
        plt.show()


In [ ]:
#MLP RUL with best hyperparameters

X = df_rul_ext[feature_cols_ext].values.astype(np.float32)
y = df_rul_ext['rul'].values.astype(np.float32)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.20, random_state=42)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=42)

train_ds = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
test_ds = TensorDataset(torch.tensor(X_te), torch.tensor(y_te))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds,batch_size=64, shuffle=False)

model_best = MLP_RUL(in_dim=len(feature_cols_ext), hidden=256, drop=0.1).to(device)
optimizer = optim.AdamW(model_best.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.L1Loss()

train_maes, val_maes = [], []

for epoch in range(1, 61):

    model_best.train()
    total, count = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model_best(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * xb.size(0)
        count += xb.size(0)
    train_mae = total / count
    train_maes.append(train_mae)

    model_best.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            total += criterion(model_best(xb), yb).item() * xb.size(0)
            count += xb.size(0)
    val_mae = total / count
    val_maes.append(val_mae)

    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:02d} | Train MAE: {train_mae:.4f} | Val MAE: {val_mae:.4f}")

model_best.eval()
preds, trues = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        preds.append(model_best(xb).cpu().numpy())
        trues.append(yb.numpy())
y_pred = np.concatenate(preds)
y_true = np.concatenate(trues)

test_mae = mean_absolute_error(y_true, y_pred)
test_mae_pct = test_mae / y_true.mean() * 100
print(f"\nTest MAE: {test_mae:.1f} cycles ({test_mae_pct:.1f}%)")

plt.figure(figsize=(5,5))
plt.scatter(y_true, y_pred, alpha=0.6)
lims = [0, max(y_true.max(), y_pred.max())]
plt.plot(lims, lims, 'k--')
plt.xlabel("True RUL (cycles)")
plt.ylabel("Predicted RUL (cycles)")
plt.title("MLP (best hyperparams): True vs Predicted RUL")
plt.tight_layout()
plt.show()

In [ ]:
mean_rul = np.mean(y_tr)  
epochs = np.arange(1, len(train_maes) + 1)
train_pct = np.array(train_maes) / mean_rul * 100
val_pct = np.array(val_maes) / mean_rul * 100

plt.figure(figsize=(5,4))
plt.plot(epochs, train_pct, label="Train MAE %")
plt.plot(epochs, val_pct, label="Val MAE %")
plt.xlabel("Epoch")
plt.ylabel("MAE (% of avg RUL)")
plt.title("Training / Validation MAE (percent)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#SHAP Feature Importance, how much change in feature changes the prediction
val_tensor = torch.tensor(X_val, dtype=torch.float32, device=device)
raw_shap = explainer.shap_values(val_tensor)
shap_vals = raw_shap[0] if isinstance(raw_shap, list) else raw_shap

plt.figure(figsize=(6,3))
feat_names = feature_cols
mean_abs = np.mean(np.abs(shap_vals), axis=0)
plt.bar(feat_names, mean_abs, edgecolor='k')
plt.xticks(rotation=45, ha='right')
plt.ylabel("Mean |SHAP|")
plt.title("Global Feature Importance (MLP RUL)")
plt.tight_layout()
plt.show()

shap.summary_plot(shap_vals, X_val, feature_names=feat_names, plot_type="dot")

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x='stage', y='resid', data=df_res, palette='Set2')
plt.axhline(0, color='k', linestyle='--')
plt.xlabel("Life Stage")
plt.ylabel("Residual (pred – true) [cycles]")
plt.title("Residuals by Battery Stage")
plt.tight_layout()
plt.show()